# 02 - Dataset Finalization
Clean, deduplicate, and split the dataset. Outputs go to `data/processed/`.

In [1]:
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent))
import pandas as pd, json
from src.data.loading import load_raw
from src.data.cleaning import clean_dataset
from src.data.splitting import make_splits, SEED
OUT = pathlib.Path('../data/processed')

In [2]:
raw = load_raw()
clean, log = clean_dataset(raw)
print(json.dumps(log, indent=2))
print(clean['label'].value_counts(normalize=True).round(3))

Skipped (not raw labeled email data): ['spamResults']
{
  "start": 82675,
  "after_drop_missing_label": 82675,
  "after_drop_empty_text": 82675,
  "after_drop_conflicting_labels": 82675,
  "after_exact_dedup": 82080
}
label
1    0.522
0    0.478
Name: proportion, dtype: float64


### Cleaning log
Copy the numbers above into the Week 1 journal entry (rows removed at each step).

In [3]:
train, val, test = make_splits(clean)
for name, d in [('train', train), ('val', val), ('test', test)]:
    d.to_csv(OUT / f'{name}.csv', index=False)
    print(name, d.shape, d['label'].mean().round(3))
clean.to_csv(OUT / 'dataset_clean.csv', index=False)

train (57456, 5) 0.522
val (12312, 5) 0.522
test (12312, 5) 0.522


### Optional: leakage check
Exact duplicates are removed, but near-duplicates (e.g. mass-mailed templates) may still cross splits. Quick check via a normalized prefix:

In [4]:
key = lambda s: s.str.lower().str[:100]
overlap = set(key(train['text'])) & set(key(test['text']))
print('train/test near-duplicate prefixes:', len(overlap))

train/test near-duplicate prefixes: 963


### Optional: source-held-out split (used in Week 4 generalization analysis)
Saves per-source membership so a model can be trained on some sources and tested on another.

In [5]:
print(clean['source'].value_counts())
clean[['source']].to_csv(OUT / 'source_index.csv')

source
CEAS_08              38963
Enron                29423
SpamAssasin           5802
Nigerian_Fraud        3293
Ling                  2859
Nazario               1552
PhishingEmailData      188
Name: count, dtype: int64


**Finalized dataset saved.** Next: Week 2 - EDA and candidate engineered features.